# GPU Profiling: Base vs Control vs Runtime-Aware

Profiles all three models with the same setup (same hardware, batch size, and input) using PyTorch Profiler.
Fine-tuned adapters are merged into the base model before profiling.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get('EFFICIENT_CODEGEN_ROOT', '/workspace/efficient-codegen'))
os.chdir(PROJECT_ROOT)

BASE_MODEL      = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
CONTROL_ADAPTER = str(PROJECT_ROOT / 'checkpoints/control')
RUNTIME_ADAPTER = str(PROJECT_ROOT / 'checkpoints/runtime_aware')

CONTROL_MERGED  = str(PROJECT_ROOT / 'checkpoints/control_merged')
RUNTIME_MERGED  = str(PROJECT_ROOT / 'checkpoints/runtime_aware_merged')

PROFILE_INPUT   = 'data/curated/prototyping/prototype_final_20_clean.json'
BATCH_SIZE      = 20
MAX_NEW_TOKENS  = 128
LIMIT           = 20

WANDB_PROJECT   = 'hpml-efficient-codegen'
WANDB_ENTITY    = 'efficient-codegen'
USE_WANDB       = True

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BASE_MODEL:  ', BASE_MODEL)
print('CONTROL_ADAPTER:', CONTROL_ADAPTER)
print('RUNTIME_ADAPTER:', RUNTIME_ADAPTER)
print('USE_WANDB:', USE_WANDB)


## Step 1: Merge LoRA Adapters

Fine-tuned checkpoints are LoRA adapters — merge them into the base model weights before profiling.
Skip if merged checkpoints already exist.

In [ ]:
def merge_adapter(adapter_path, output_dir):
    if Path(output_dir).exists():
        print(f'Already merged: {output_dir}')
        return
    print(f'Merging {adapter_path} -> {output_dir}')
    subprocess.run([
        sys.executable, 'serving/merge_checkpoint.py',
        '--adapter_path',    adapter_path,
        '--base_model_name', BASE_MODEL,
        '--output_dir',      output_dir,
    ], check=True)
    print('Done.')

merge_adapter(CONTROL_ADAPTER, CONTROL_MERGED)
merge_adapter(RUNTIME_ADAPTER, RUNTIME_MERGED)

## Step 2: Profile Each Model

Runs `profile_model.py` for each model with identical settings.
Traces are saved to `outputs/gpu_profiling/<model>/`.

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
MODELS = {
    'base':          BASE_MODEL,
    'control':       CONTROL_MERGED,
    'runtime_aware': RUNTIME_MERGED,
}

TRACE_BASE = PROJECT_ROOT / 'outputs/gpu_profiling'

summaries = {}

for label, model_path in MODELS.items():
    trace_dir = str(TRACE_BASE / label)
    print(f'\n{"="*60}')
    print(f'Profiling: {label}')
    print(f'{"="*60}')
    cmd = [
        sys.executable, 'profiling/profile_model.py',
        '--input_path',     PROFILE_INPUT,
        '--model_name',     model_path,
        '--trace_dir',      trace_dir,
        '--limit',          str(LIMIT),
        '--batch_size',     str(BATCH_SIZE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--wandb_project',  WANDB_PROJECT,
        '--wandb_entity',   WANDB_ENTITY,
        '--wandb_run_name', f'gpu-profiling-{label}',
    ]
    if USE_WANDB:
        cmd.append('--use_wandb')
    subprocess.run(cmd, check=True)
    summaries[label] = {'trace_dir': trace_dir}


## Step 3: Parse Traces and Compare

In [ ]:
import json
from collections import defaultdict
import pandas as pd

TOP_N = 15
REPORT = {}

for label in MODELS:
    trace_dir = TRACE_BASE / label
    trace_files = sorted(trace_dir.rglob('*.pt.trace.json')) if trace_dir.exists() else []
    if not trace_files:
        print(f'No traces found for {label}')
        continue

    trace_file = max(trace_files, key=lambda f: f.stat().st_size)
    print(f'\n[{label}] Parsing: {trace_file.name}')

    with open(trace_file) as f:
        data = json.load(f)

    events = data.get('traceEvents', data) if isinstance(data, dict) else data
    op_time = defaultdict(float)
    op_calls = defaultdict(int)
    total = 0.0
    for e in events:
        if not isinstance(e, dict):
            continue
        if e.get('ph') == 'X' and e.get('dur', 0) > 0:
            name = e.get('name', 'unknown')
            dur_ms = e['dur'] / 1000
            op_time[name] += dur_ms
            op_calls[name] += 1
            total += dur_ms

    top = sorted(op_time.items(), key=lambda x: x[1], reverse=True)[:TOP_N]
    REPORT[label] = {'total_ms': total, 'top_ops': top, 'op_time': op_time, 'op_calls': op_calls}

    print(f"  Total traced time: {total:.2f} ms")
    print(f"  {'Operator':<50} {'ms':>10} {'%':>7}")
    print(f"  {'-'*70}")
    for name, ms in top:
        pct = ms / total * 100
        print(f"  {name:<50} {ms:>10.2f} {pct:>6.1f}%")

## Step 4: Side-by-Side Comparison

Top operators compared across all three models.

In [ ]:
KEY_OPS = [
    'aten::linear',
    'aten::scaled_dot_product_attention',
    'aten::matmul',
    'aten::mm',
    'aten::addmm',
    'aten::mul',
    'aten::add',
    'aten::item',
    'cudaStreamSynchronize',
    'cudaLaunchKernel',
]

rows = []
for op in KEY_OPS:
    row = {'Operator': op}
    for label in MODELS:
        if label not in REPORT:
            row[f'{label} ms'] = None
            row[f'{label} %'] = None
            continue
        ms = REPORT[label]['op_time'].get(op, 0.0)
        pct = ms / REPORT[label]['total_ms'] * 100
        row[f'{label} ms'] = round(ms, 2)
        row[f'{label} %'] = round(pct, 2)
    rows.append(row)

df = pd.DataFrame(rows).set_index('Operator')
print('Key Operator Comparison (ms and % of total traced time)')
print('=' * 80)
df

## Step 5: TensorBoard

View traces for all three models side by side in TensorBoard.
Each model's traces are in a separate subdirectory, which TensorBoard uses as the run label.

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir outputs/gpu_profiling

## Step 6: Operator-Level Profiling

Runs `profile_operators.py` for each model, producing a bottleneck report and CSV of operator times.
Output is saved to `outputs/operator_profiling/<model>/`.

In [ ]:
OP_PROFILE_BASE = PROJECT_ROOT / 'outputs/operator_profiling'

for label, model_path in MODELS.items():
    output_dir = str(OP_PROFILE_BASE / label)
    print(f'\n{"="*60}')
    print(f'Operator profiling: {label}')
    print(f'{"="*60}')
    subprocess.run([
        sys.executable, 'profiling/profile_operators.py',
        '--input_path',     PROFILE_INPUT,
        '--model_name',     model_path,
        '--limit',          str(LIMIT),
        '--batch_size',     str(BATCH_SIZE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--output_dir',     output_dir,
    ], check=True)


## Step 7: Operator Bottleneck Reports

In [ ]:
for label in MODELS:
    report = OP_PROFILE_BASE / label / 'bottleneck_report.txt'
    if report.exists():
        print(f'\n{"="*60}')
        print(f'  {label}')
        print(f'{"="*60}')
        print(report.read_text(encoding='utf-8')[:3000])
    else:
        print(f'No report found for {label}')


## Step 8: Operator CSV Comparison

Loads `operators.csv` from each model run and shows a side-by-side comparison of top operators by CPU time.

In [ ]:
import pandas as pd

op_dfs = {}
for label in MODELS:
    csv_path = OP_PROFILE_BASE / label / 'operators.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        op_dfs[label] = df.set_index('name') if 'name' in df.columns else df
    else:
        print(f'No operators.csv for {label}')

if op_dfs:
    col = 'cpu_time_total'
    top_ops = (
        pd.concat({k: v[col] for k, v in op_dfs.items() if col in v.columns}, axis=1)
        .fillna(0)
        .assign(total=lambda d: d.sum(axis=1))
        .sort_values('total', ascending=False)
        .drop(columns='total')
        .head(20)
    )
    print('Top 20 operators by CPU time (ms)')
    print('=' * 60)
    top_ops / 1000
